## Inventory Diff Report Table

- Help us create a table to compute the difference between the simulation initial conditions and the inventory data

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
OUTPUT_POSTPROCESSING_DIR_PATH = os.getcwd()
SSP_MODELING_DIR_PATH = os.path.dirname(OUTPUT_POSTPROCESSING_DIR_PATH)
SSP_OUTPUT_DIR_PATH = os.path.join(SSP_MODELING_DIR_PATH, "ssp_run_output")
CW_DATA_DIR_PATH = os.path.join(OUTPUT_POSTPROCESSING_DIR_PATH, "data")

In [3]:
ISO3 = "BGR"
REGION_NAME = "bulgaria"
RUN_DIR_PATH = os.path.join(SSP_OUTPUT_DIR_PATH, "sisepuede_results_sisepuede_run_2025-12-12T12;40;03.133540")

### Load emission targets and ssp outputs dfs

In [4]:
# Load emission targets
emission_targets_df = pd.read_csv(os.path.join(CW_DATA_DIR_PATH, "emission_targets_bulgaria_2022.csv"))
emission_targets_df.head()

,Subsector,Gas,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,id,BGR,Edgar_Class
0,agrc,CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops - CH4,0.125150,AG - Crops:CH4
1,agrc,CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_soil_soc_mineral_soils:emiss...,AG - Crops - CO2,0.094230,AG - Crops:CO2
2,agrc,N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_soil_fertilizer:emission_co2...,AG - Crops - N2O,3.402720,AG - Crops:N2O
3,lvst,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock - CH4,1.976805,AG - Livestock:CH4
4,lsmm,N2O,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_n2o_lsmm_direct_anaerobic_digest...,AG - Livestock - N2O,0.306905,AG - Livestock:N2O


In [5]:
# Load output data
ssp_output_df = pd.read_csv(os.path.join(RUN_DIR_PATH, 
                                         "sisepuede_results_sisepuede_run_2025-12-12T12;40;03.133540_WIDE_INPUTS_OUTPUTS.csv"))
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,bulgaria,0,0.0,1.885187e+06,4530.325383,77118.918650,40648.867064,7620.497590,2.386984e+06,...,1.039938,0.839997,1.062913,0.5,1.966460,5.4540,55.765409,14.9633,2.555236,92.81
1,0,bulgaria,1,0.0,1.885342e+06,4530.699868,77125.293436,40652.227172,7621.127514,2.387182e+06,...,1.051730,0.872306,1.204477,0.5,2.296954,5.4032,55.765409,15.1841,2.967399,92.81
2,0,bulgaria,2,0.0,1.891529e+06,4545.567908,77378.389423,40785.632377,7646.137167,2.395015e+06,...,1.011737,0.821764,1.196420,0.5,2.479671,5.6089,55.765409,17.7897,2.826425,92.81
3,0,bulgaria,3,0.0,1.888686e+06,4538.734259,77262.061440,40724.316673,7634.642229,2.391415e+06,...,1.011737,0.842645,2.481176,0.5,1.299027,5.5200,55.765409,18.5596,3.076353,92.81
4,0,bulgaria,4,0.0,1.878172e+06,4513.468170,76831.961333,40497.613779,7592.141934,2.378102e+06,...,1.011737,0.693288,2.747621,0.5,2.005811,6.0000,55.765409,21.2497,2.942655,92.81


### Obtain the ssp output values in the emission targets format

In [6]:
def sum_vars_from_ssp_outputs(
    emission_targets_df: pd.DataFrame,
    ssp_outputs_df: pd.DataFrame,
    vars_col: str = "Vars",
    out_col: str = "ssp_total",
    record_missing_col: str | None = "missing_vars",
    ssp_filter: dict | None = None,
) -> pd.DataFrame:
    """
    For each row in emission_targets_df, split the colon-separated strings in `vars_col`,
    find those columns in ssp_outputs_df, sum their values (over rows & columns), and
    write the total to `out_col` in emission_targets_df.

    Parameters
    ----------
    emission_targets_df : DataFrame
        Must contain a string column `vars_col` with colon-separated names.
    ssp_outputs_df : DataFrame
        Wide table whose columns include the names referenced by `emission_targets_df[vars_col]`.
    vars_col : str
        Column in emission_targets_df with colon-separated variable names.
    out_col : str
        New column to create in emission_targets_df with totals from ssp_outputs_df.
    record_missing_col : str | None
        If provided, creates a column listing any missing vars for each row.
    df2_filter : dict | None
        Optional filters to reduce ssp_outputs_df before summing, e.g.
        {"region": "egypt", "time_period": 7}

    Returns
    -------
    DataFrame
        emission_targets_df with new column `out_col` (and `record_missing_col` if requested).
    """
    # Optionally filter ssp_outputs_df by key=value pairs (e.g., region/time_period)
    if ssp_filter:
        mask = pd.Series(True, index=ssp_outputs_df.index)
        for k, v in ssp_filter.items():
            mask &= (ssp_outputs_df[k] == v)
        ssp_view = ssp_outputs_df.loc[mask]
    else:
        ssp_view = ssp_outputs_df

    # Ensure we only operate on numeric data when summing
    numeric_cols = set(ssp_view.select_dtypes(include=[np.number]).columns)

    def _total_for_vars(vars_str: str):
        if pd.isna(vars_str) or not str(vars_str).strip():
            return np.nan, []

        # Split, strip, and deduplicate while preserving order
        raw = [s.strip() for s in str(vars_str).split(":") if s.strip()]
        seen = set()
        cols = [c for c in raw if not (c in seen or seen.add(c))]

        present = [c for c in cols if c in ssp_view.columns and c in numeric_cols]
        missing = [c for c in cols if c not in ssp_view.columns or c not in numeric_cols]

        if not present or ssp_view.empty:
            return np.nan, missing

        # Sum over all filtered rows & all present columns
        vals = ssp_view[present].to_numpy(dtype=float, copy=False)
        total = np.nansum(vals)
        return float(total), missing

    totals, missings = [], []
    for v in emission_targets_df[vars_col].astype("string"):
        total, missing = _total_for_vars(v)
        totals.append(total)
        missings.append(missing)

    emission_targets_df = emission_targets_df.copy()
    emission_targets_df[out_col] = totals
    if record_missing_col is not None:
        emission_targets_df[record_missing_col] = missings

    return emission_targets_df


# -----------------------------
# Example usage
# -----------------------------

# If DF2 has a single row for the target (e.g., region="egypt", a specific time_period):
# df2_filter = {"region": "egypt"}              # or {"region": "egypt", "time_period": 7}
# If you want to sum across all rows of DF2, set df2_filter = None.

# df1_result = sum_vars_from_df2(DF1, DF2, vars_col="Vars",
#                                out_col="DF2_total",
#                                record_missing_col="Missing_in_DF2",
#                                df2_filter={"region": "egypt"})
# print(df1_result.head())


In [7]:
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,bulgaria,0,0.0,1.885187e+06,4530.325383,77118.918650,40648.867064,7620.497590,2.386984e+06,...,1.039938,0.839997,1.062913,0.5,1.966460,5.4540,55.765409,14.9633,2.555236,92.81
1,0,bulgaria,1,0.0,1.885342e+06,4530.699868,77125.293436,40652.227172,7621.127514,2.387182e+06,...,1.051730,0.872306,1.204477,0.5,2.296954,5.4032,55.765409,15.1841,2.967399,92.81
2,0,bulgaria,2,0.0,1.891529e+06,4545.567908,77378.389423,40785.632377,7646.137167,2.395015e+06,...,1.011737,0.821764,1.196420,0.5,2.479671,5.6089,55.765409,17.7897,2.826425,92.81
3,0,bulgaria,3,0.0,1.888686e+06,4538.734259,77262.061440,40724.316673,7634.642229,2.391415e+06,...,1.011737,0.842645,2.481176,0.5,1.299027,5.5200,55.765409,18.5596,3.076353,92.81
4,0,bulgaria,4,0.0,1.878172e+06,4513.468170,76831.961333,40497.613779,7592.141934,2.378102e+06,...,1.011737,0.693288,2.747621,0.5,2.005811,6.0000,55.765409,21.2497,2.942655,92.81


In [8]:
emission_targets_df_extended = sum_vars_from_ssp_outputs(emission_targets_df, ssp_output_df, vars_col="Vars",
                               out_col="ssp_emission",
                               record_missing_col="missing_in_ssp_outputs",
                               ssp_filter={"region": REGION_NAME, "primary_id": 0, "time_period": 7})

emission_targets_df_extended.head()

,Subsector,Gas,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,id,BGR,Edgar_Class,ssp_emission,missing_in_ssp_outputs
0,agrc,CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops - CH4,0.125150,AG - Crops:CH4,0.067084,[]
1,agrc,CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_soil_soc_mineral_soils:emiss...,AG - Crops - CO2,0.094230,AG - Crops:CO2,0.105592,[]
2,agrc,N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_soil_fertilizer:emission_co2...,AG - Crops - N2O,3.402720,AG - Crops:N2O,3.502606,[]
3,lvst,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock - CH4,1.976805,AG - Livestock:CH4,2.013917,[]
4,lsmm,N2O,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_n2o_lsmm_direct_anaerobic_digest...,AG - Livestock - N2O,0.306905,AG - Livestock:N2O,0.315271,[]


### Create diff report

In [9]:
# subset the emission targets to create the diff report template
diff_report_df = emission_targets_df_extended[[
    "Subsector",
    "Edgar_Class",
    ISO3,
    "ssp_emission",
]].copy()
diff_report_df.head()

,Subsector,Edgar_Class,BGR,ssp_emission
0,agrc,AG - Crops:CH4,0.125150,0.067084
1,agrc,AG - Crops:CO2,0.094230,0.105592
2,agrc,AG - Crops:N2O,3.402720,3.502606
3,lvst,AG - Livestock:CH4,1.976805,2.013917
4,lsmm,AG - Livestock:N2O,0.306905,0.315271


In [10]:
# merge subsector an id into a single column for clarity
diff_report_df["subsector_id"] = diff_report_df["Subsector"] + " - " + diff_report_df["Edgar_Class"]
diff_report_df = diff_report_df.drop(columns=["Subsector", "Edgar_Class"])
diff_report_df.head()

,BGR,ssp_emission,subsector_id
0,0.125150,0.067084,agrc - AG - Crops:CH4
1,0.094230,0.105592,agrc - AG - Crops:CO2
2,3.402720,3.502606,agrc - AG - Crops:N2O
3,1.976805,2.013917,lvst - AG - Livestock:CH4
4,0.306905,0.315271,lsmm - AG - Livestock:N2O


In [11]:
#rename region column
diff_report_df = diff_report_df.rename(columns={ISO3: "inventory_emission"})
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id
0,0.125150,0.067084,agrc - AG - Crops:CH4
1,0.094230,0.105592,agrc - AG - Crops:CO2
2,3.402720,3.502606,agrc - AG - Crops:N2O
3,1.976805,2.013917,lvst - AG - Livestock:CH4
4,0.306905,0.315271,lsmm - AG - Livestock:N2O


In [12]:
# Create inventory_share column
diff_report_df["inventory_share"] = diff_report_df["inventory_emission"] / diff_report_df["inventory_emission"].sum()
diff_report_df

,inventory_emission,ssp_emission,subsector_id,inventory_share
0,0.125150,0.067084,agrc - AG - Crops:CH4,2.499030e-03
1,0.094230,0.105592,agrc - AG - Crops:CO2,1.881611e-03
2,3.402720,3.502606,agrc - AG - Crops:N2O,6.794646e-02
3,1.976805,2.013917,lvst - AG - Livestock:CH4,3.947339e-02
4,0.306905,0.315271,lsmm - AG - Livestock:N2O,6.128364e-03
5,0.000000,0.000000,ccsq - CCSQ:CH4,0.000000e+00
6,0.000000,0.000000,ccsq - CCSQ:CO2,0.000000e+00
7,0.000000,0.000000,ccsq - CCSQ:N2O,0.000000e+00
8,0.415001,0.015267,scoe - EN - Building:CH4,8.286861e-03
9,1.695736,1.686396,scoe - EN - Building:CO2,3.386093e-02


In [13]:
# Calculate error column, avoid division by zero by adding a small constant to the denominator
epsilon = 1e-8
diff_report_df["rel_error"] = (diff_report_df["ssp_emission"] - diff_report_df["inventory_emission"]).abs() / (diff_report_df["inventory_emission"] + epsilon)
diff_report_df["abs_error"] = (diff_report_df["ssp_emission"] - diff_report_df["inventory_emission"]).abs()
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id,inventory_share,rel_error,abs_error
0,0.125150,0.067084,agrc - AG - Crops:CH4,0.002499,0.463973,0.058066
1,0.094230,0.105592,agrc - AG - Crops:CO2,0.001882,0.120578,0.011362
2,3.402720,3.502606,agrc - AG - Crops:N2O,0.067946,0.029355,0.099886
3,1.976805,2.013917,lvst - AG - Livestock:CH4,0.039473,0.018774,0.037112
4,0.306905,0.315271,lsmm - AG - Livestock:N2O,0.006128,0.027258,0.008366


In [14]:
# Set subsector_id at the beginning of the df
diff_report_df = diff_report_df[[
    "subsector_id",
    "inventory_emission",
    "ssp_emission",
    "inventory_share",
    "abs_error",
    "rel_error"
]]

# sort by squared_error descending
diff_report_df = diff_report_df.sort_values(by="rel_error", ascending=False)
diff_report_df.head(10)

,subsector_id,inventory_emission,ssp_emission,inventory_share,abs_error,rel_error
26,ippu - IN - Industrial Processes:CH4,0.000000,0.007865,0.000000,0.007865,786453.612174
34,frst - LULUCF - Forest Land:CH4,0.000000,0.001097,0.000000,0.001097,109731.606551
30,ippu - IN - Industrial Processes:OTHER_FCS,0.000000,0.000229,0.000000,0.000229,22902.722533
36,lndu - LULUCF - Wetlands:CH4,0.000000,0.000087,0.000000,0.000087,8657.385364
47,trww - Waste - Wastewater Treatment:N2O,0.000430,0.028653,0.000009,0.028223,65.634214
19,inen - EN - Manufacturing/Construction:N2O,0.023925,0.070793,0.000478,0.046868,1.958962
14,fgtv - EN - Fugitive Emissions:CH4,1.125722,2.627884,0.022479,1.502162,1.334399
17,inen - EN - Manufacturing/Construction:CH4,0.018078,0.041254,0.000361,0.023176,1.282018
18,inen - EN - Manufacturing/Construction:CO2,4.932056,10.928562,0.098485,5.996506,1.215823
8,scoe - EN - Building:CH4,0.415001,0.015267,0.008287,0.399734,0.963212


In [15]:
diff_report_df.tail(40)

,subsector_id,inventory_emission,ssp_emission,inventory_share,abs_error,rel_error
18,inen - EN - Manufacturing/Construction:CO2,4.932056,10.928562,9.848466e-02,5.996506,1.215823
8,scoe - EN - Building:CH4,0.415001,0.015267,8.286861e-03,0.399734,0.963212
31,ippu - IN - Industrial Processes:PFC,0.000010,0.000019,1.996828e-07,0.000009,0.929655
23,trns - EN - Transportation:CH4,0.050660,0.006344,1.011589e-03,0.044316,0.874778
11,entc - EN - Electricity/Heat:CH4,0.023684,0.004073,4.729268e-04,0.019611,0.828024
24,trns - EN - Transportation:CO2,10.314811,2.500370,2.059690e-01,7.814441,0.757594
10,scoe - EN - Building:N2O,0.087103,0.027629,1.739302e-03,0.059474,0.682803
15,fgtv - EN - Fugitive Emissions:CO2,2.463175,0.897216,4.918537e-02,1.565960,0.635748
16,fgtv - EN - Fugitive Emissions:N2O,0.000856,0.001391,1.709279e-05,0.000535,0.624695
13,entc - EN - Electricity/Heat:N2O,0.106404,0.040141,2.124712e-03,0.066264,0.622753


### Save diff table

In [16]:
diff_report_df.to_clipboard(index=False)

In [16]:
diff_report_df.to_csv(os.path.join(RUN_DIR_PATH, f"inventory_diff_report_{REGION_NAME}.csv"), index=False)